In [360]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

from sklearn.model_selection import train_test_split

#for comparisions
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDRegressor 

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [361]:
X, y = load_diabetes(return_X_y = True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

In [ ]:
class My_Ridge_SGD:
    def __init__(self, learning_rate: float, max_iter: int, lambda_: float = 0,) -> None:
        self.iter = max_iter
        self.learning_rate = learning_rate
        self.coef_ = np.array([])
        self.intercept_ = 0
        self.lambda_ = lambda_
        self.weights = np.array([])

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.array(y)
        ones = np.ones((X.shape[0], 1))
        X = np.hstack((ones, X))
        rows, columns = X.shape

        self.weights = np.random.rand(columns)
        for i in range(self.iter):
            for _ in range(rows):
                idx = np.random.randint(low = 0, high = rows)
                x_random = X[idx]
                y_random = y[idx]
                
                alpha_derivative = -2 * (self.lambda_ * self.weights)
                
                #no penalty for the intercept
                alpha_derivative[0] = 0

                error = y_random - (x_random @ self.weights)

                gradient = -2 * error * x_random + alpha_derivative
                self.weights -= self.learning_rate * gradient
        print(self.weights)

    def predict(self, X):
        X = np.asarray(X)
        ones = np.ones((X.shape[0], 1))
        X = np.hstack((ones, X))
        return X @ self.weights

def my_r2_score(y_true, y_pred):
    SSM = np.sum((y_true - np.mean(y_true)) ** 2)
    SSR = np.sum((y_true - y_pred) ** 2)
    R2 = 1 - SSR/SSM
    return R2


In [ ]:
ridge1 = My_Ridge_SGD(lambda_ = 0.08858667904100796, max_iter = 10, learning_rate = 0.004)
ridge1.fit(X = X_train, y = y_train)
print(f'My r2 score: {my_r2_score(y_test, ridge1.predict(X_test))}')
my_r2_score(y_test, ridge1.predict(X_test))

[ 148.92161619   88.80052532   26.55578303  266.02723736  195.26322396
   77.87168625   68.15470706 -141.94135911  146.05912474  226.88396912
  147.51803137]
My r2 score: 0.3707254944055317


0.3707254944055317

In [ ]:
ridge2 = SGDRegressor(loss="squared_error", penalty="l2", alpha = 4.281332398719396e-08, learning_rate = 'constant')
ridge2.fit(X = X_train, y = y_train)
print(ridge2.intercept_, ridge2.coef_)
print(f'Scikit Learn r2 score: {my_r2_score(y_test, ridge2.predict(X_test))}')

[152.96969677] [  50.70748275  -85.29065471  379.17569569  265.17850404    7.83309337
  -44.61701487 -180.88035698  127.42088454  343.67356939  127.78817567]
Scikit Learn r2 score: 0.44134739794588096


In [365]:
print(f'My cross_val_score: {cross_val_score(ridge2, X_train, y_train, cv = 10, scoring = 'r2').mean()}')

My cross_val_score: 0.4632063425494698


In [366]:
params = {
    'alpha' : np.logspace(-20, 20, 20)
}

grid = GridSearchCV(SGDRegressor(loss="squared_error", penalty="l2", alpha = 0.08858667904100796,), param_grid=params, cv = 5, scoring = 'r2', )
grid.fit(X_train, y_train)

print("Best alpha:", grid.best_params_)
print("Best score:", grid.best_score_)

/home/vhvhs/ML/.venv/lib64/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/vhvhs/ML/.venv/lib64/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/vhvhs/ML/.venv/lib64/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/vhvhs/ML/.venv/lib64/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/vhvhs/ML/.venv

Best alpha: {'alpha': np.float64(2.6366508987303555e-12)}
Best score: 0.39100400559595716


/home/vhvhs/ML/.venv/lib64/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/vhvhs/ML/.venv/lib64/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
